# Eco-Counter and Vivacity — API fetch and analysis

This notebook mirrors the Dublin Active Travel Dashboard backend:

- **Eco-Counter**: same REST endpoints and headers as `+page.server.js` and `api/eco-counter/timeseries`, plus `static/counter_activity.json` for active-site filtering (as in the app).
- **Vivacity**: `hardware/metadata`, `countline/counts` with the same date windows and aggregation as `api/vivacity-counter/timeseries`, plus `static/vivacity_markers.json` and metadata merge from `+page.server.js`.

**Setup**

1. Set environment variables (or a `.env` file in this project root):
   - `ECO_COUNTER_API` — Eco-Counter API key
   - `VIVACITY_API` — Vivacity API key
2. Optional: `pip install requests pandas python-dotenv`
3. Run cells top to bottom.

No charts — only tabular summaries and printed stats.

In [1]:
# Optional: install dependencies in the active Jupyter kernel
# %pip install -q requests pandas python-dotenv

In [2]:
import json
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
import requests

try:
    from dotenv import load_dotenv
    load_dotenv(Path(".env"))
except ImportError:
    pass

ECO_KEY = os.environ.get("ECO_COUNTER_API")
VIVACITY_KEY = os.environ.get("VIVACITY_API")

ROOT = Path(".").resolve()
STATIC = ROOT / "static"

if not ECO_KEY:
    raise ValueError("Set ECO_COUNTER_API in the environment (or .env)")
if not VIVACITY_KEY:
    print("Warning: VIVACITY_API not set — Vivacity sections will be skipped.")

SESSION = requests.Session()
SESSION.headers.update({"Accept": "application/json"})

## Eco-Counter — fetch (dashboard load)

Matches `+page.server.js`: sites list + statistical ADT by site/travel mode.

In [3]:
def eco_headers():
    return {"accept": "application/json", "X-API-KEY": ECO_KEY}


def fetch_eco_sites():
    url = "https://api.eco-counter.com/api/v2/sites?page=1&pageSize=100&sortBy=id&orderBy=asc"
    r = SESSION.get(url, headers=eco_headers(), timeout=120)
    r.raise_for_status()
    return r.json()


def fetch_eco_traffic_adt():
    url = (
        "https://api.eco-counter.com/api/v2/statistical/adt/by/site"
        "?dateRange=lastMonth&groupBy=siteAndTravelMode"
        "&travelModes=pedestrian&travelModes=bike"
    )
    r = SESSION.get(url, headers=eco_headers(), timeout=120)
    r.raise_for_status()
    return r.json()


eco_sites_raw = fetch_eco_sites()
eco_traffic_raw = fetch_eco_traffic_adt()

print("Eco sites:")
if isinstance(eco_sites_raw, dict):
    print("  type: dict, keys:", list(eco_sites_raw.keys()))
else:
    print("  type: list, length:", len(eco_sites_raw))

print("Eco ADT traffic:")
tr = eco_traffic_raw
print("  type:", type(tr).__name__)
if isinstance(tr, dict):
    print("  keys:", list(tr.keys())[:30])
elif isinstance(tr, list):
    print("  list length:", len(tr))
    if tr:
        el0 = tr[0]
        print("  [0] type:", type(el0).__name__)
        if isinstance(el0, dict):
            print("  [0] keys:", list(el0.keys())[:40])
else:
    print("  repr:", repr(tr)[:300])


Eco sites:
  type: list, length: 88
Eco ADT traffic:
  type: list
  list length: 108
  [0] type: dict
  [0] keys: ['siteId', 'travelMode', 'value']


### Eco-Counter — static `counter_activity.json` + processor logic

Same shape as `processEcoCounterLocations`, `processEcoCounterTraffic`, `combineEcoCounterData` in `eco-counter-processor.js`, then filter **active** sites using `counter_activity.json` like the app.

In [4]:
import json

def load_json(path: Path):
    if not path.is_file():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _eco_int_id(v):
    if v is None:
        return None
    try:
        return int(v)
    except (TypeError, ValueError):
        return v


counter_activity = load_json(STATIC / "counter_activity.json") or []
activity_by_site = {}
for row in counter_activity:
    if "site_id" not in row:
        continue
    k = _eco_int_id(row["site_id"])
    if k is not None:
        activity_by_site[k] = row


ECO_DEBUG = True  # set False to hide nested-structure logs


def collect_numeric_paths(obj, prefix="", depth=0, max_depth=14, out=None):
    """All numeric leaves with dotted paths (for debugging nested ADT)."""
    if out is None:
        out = []
    if depth > max_depth:
        return out
    if isinstance(obj, dict):
        for k, v in obj.items():
            seg = f"{prefix}.{k}" if prefix else str(k)
            lk = str(k).lower()
            if lk in ("latitude", "longitude"):
                continue
            collect_numeric_paths(v, seg, depth + 1, max_depth, out)
    elif isinstance(obj, list):
        for i, v in enumerate(obj[:30]):
            collect_numeric_paths(v, f"{prefix}[{i}]", depth + 1, max_depth, out)
    elif isinstance(obj, (int, float)) and not isinstance(obj, bool):
        out.append((prefix, float(obj)))
    elif isinstance(obj, str):
        s = obj.replace(",", "").strip()
        try:
            out.append((prefix, float(s)))
        except ValueError:
            pass
    return out


def unwrap_eco_traffic_list(traffic_data):
    if traffic_data is None:
        return []
    if isinstance(traffic_data, list):
        return traffic_data
    if isinstance(traffic_data, dict):
        for k in ("data", "content", "results", "items", "records", "_embedded"):
            v = traffic_data.get(k)
            if isinstance(v, list):
                return v
        return []
    return []


def _eco_site_id_from_record(t):
    for k in ("siteId", "site_id"):
        if k in t and t[k] is not None:
            return _eco_int_id(t[k])
    site = t.get("site")
    if isinstance(site, dict):
        for k in ("id", "siteId", "site_id"):
            if k in site and site[k] is not None:
                return _eco_int_id(site[k])
    # Site-grouped ADT row: { "id": <siteId>, "data": [ {mode stats}, ... ] }
    nested_keys = ("data", "travelStatistics", "statistics", "travelModes", "byTravelMode", "values")
    if any(isinstance(t.get(k), list) for k in nested_keys):
        if t.get("id") is not None:
            return _eco_int_id(t["id"])
    return None


def expand_traffic_rows(traffic_list):
    """Flatten nested site + per-mode rows (some groupBy responses)."""
    out = []
    for t in traffic_list:
        if not isinstance(t, dict):
            continue
        nested = (
            t.get("travelStatistics")
            or t.get("statistics")
            or t.get("travel_modes")
            or t.get("byTravelMode")
        )
        sid = _eco_site_id_from_record(t)
        if isinstance(nested, list) and sid is not None:
            for sub in nested:
                if isinstance(sub, dict):
                    row = dict(sub)
                    row.setdefault("siteId", sid)
                    out.append(row)
            continue
        out.append(t)
    return out



def _looks_like_flat_adt_row(d):
    if not isinstance(d, dict):
        return False
    has_site = d.get("siteId") is not None or d.get("site_id") is not None
    has_site = has_site or (isinstance(d.get("site"), dict) and d["site"].get("id") is not None)
    has_mode = (
        d.get("travelMode") is not None
        or d.get("travel_mode") is not None
        or d.get("mode") is not None
        or d.get("userType") is not None
    )
    return has_site and has_mode


def normalize_statistical_adt_list(traffic_list):
    """Expand /statistical/adt/by/site list-of-sites into flat per-mode rows."""
    if not traffic_list or not isinstance(traffic_list, list):
        return traffic_list
    if _looks_like_flat_adt_row(traffic_list[0]):
        return traffic_list
    expanded = []
    for item in traffic_list:
        if not isinstance(item, dict):
            continue
        sid = _eco_site_id_from_record(item)
        if sid is None:
            sid = _eco_int_id(item.get("id"))
        placed = False
        for key in (
            "data",
            "travelStatistics",
            "statistics",
            "travelModes",
            "byTravelMode",
            "values",
            "modes",
        ):
            arr = item.get(key)
            if not isinstance(arr, list) or sid is None:
                continue
            for sub in arr:
                if isinstance(sub, dict):
                    row = dict(sub)
                    row.setdefault("siteId", sid)
                    expanded.append(row)
            placed = True
            break
        if not placed:
            expanded.append(item)
    return expanded


def _eco_travel_mode(t):
    m = t.get("travelMode") or t.get("travel_mode") or t.get("mode")
    if m is None and "userType" in t:
        ut = t.get("userType")
        m = {1: "pedestrian", 2: "bike", 3: "horse", 7: "undefined", 13: "escooter"}.get(ut)
    if isinstance(m, int):
        m = {1: "pedestrian", 2: "bike", 3: "horse", 7: "undefined", 13: "escooter"}.get(m, str(m))
    return m


def _eco_adt(t, _depth=0):
    """Pick best numeric ADT-like value using path scoring over all nested numbers."""
    if not isinstance(t, dict):
        return 0.0
    paths = collect_numeric_paths(t, "row", 0, 14)
    if not paths:
        return 0.0

    def score_path(path, val):
        pl = path.lower()
        s = 0
        if any(x in pl for x in ("averagedailytraffic", "average_daily_traffic", "averagedaily", "dailyaverage")):
            s += 120
        elif "average" in pl and "daily" in pl:
            s += 100
        elif "adt" in pl:
            s += 90
        elif "average" in pl or "mean" in pl:
            s += 60
        elif "daily" in pl:
            s += 50
        elif "traffic" in pl or "statistic" in pl:
            s += 25
        elif "count" in pl or "total" in pl:
            s += 15
        if pl.endswith(".siteid") or pl.endswith(".id"):
            if val > 50000:
                s -= 200
        if "latitude" in pl or "longitude" in pl:
            s -= 200
        return s

    best_score = max(score_path(p, v) for p, v in paths)
    if best_score > 0:
        best = [pv for pv in paths if score_path(pv[0], pv[1]) == best_score]
        best.sort(key=lambda pv: -abs(pv[1]))
        return float(best[0][1])
    for path, val in sorted(paths, key=lambda pv: (-abs(pv[1]))):
        if 0 < val < 5e7 and score_path(path, val) > -80:
            return float(val)
    return 0.0

def process_eco_locations(sites_data):
    if isinstance(sites_data, list):
        sites = sites_data
    elif isinstance(sites_data, dict) and isinstance(sites_data.get("data"), list):
        sites = sites_data["data"]
    else:
        return []
    out = []
    for site in sites:
        if not isinstance(site, dict):
            continue
        loc = site.get("location") or {}
        out.append(
            {
                "id": _eco_int_id(site.get("id")),
                "name": site.get("name"),
                "latitude": loc.get("lat"),
                "longitude": loc.get("lon"),
                "travelModes": site.get("travelModes") or [],
            }
        )
    return out


def process_eco_traffic(traffic_data, sites_data):
    raw_list = unwrap_eco_traffic_list(traffic_data)
    raw_list = normalize_statistical_adt_list(raw_list)
    raw_list = expand_traffic_rows(raw_list)
    sites_map = {}
    if isinstance(sites_data, list):
        for s in sites_data:
            if isinstance(s, dict):
                i = _eco_int_id(s.get("id"))
                if i is not None:
                    sites_map[i] = s
    elif isinstance(sites_data, dict) and isinstance(sites_data.get("data"), list):
        for s in sites_data["data"]:
            if isinstance(s, dict):
                i = _eco_int_id(s.get("id"))
                if i is not None:
                    sites_map[i] = s
    rows = []
    for t in raw_list:
        if not isinstance(t, dict):
            continue
        sid = _eco_site_id_from_record(t)
        tm = _eco_travel_mode(t)
        if sid is None or tm is None:
            continue
        merged = dict(t)
        merged["siteId"] = sid
        merged["siteName"] = (sites_map.get(sid) or {}).get("name") or f"Site {sid}"
        if merged.get("travelMode") is None:
            merged["travelMode"] = tm
        merged["averageDailyTraffic"] = _eco_adt(merged)
        rows.append(merged)
    return rows


def combine_eco(locations, traffic_rows):
    by_site = {}
    for t in traffic_rows:
        sid = t["siteId"]
        tm = t["travelMode"]
        by_site.setdefault(sid, {})[tm] = t
    combined = []
    for loc in locations:
        sid = loc["id"]
        tm_map = by_site.get(sid, {}) if sid is not None else {}
        total = sum(_eco_adt(v) for v in tm_map.values())
        combined.append({**loc, "traffic_by_mode": tm_map, "total_average_daily_traffic": total})
    return combined


locations = process_eco_locations(eco_sites_raw)
traffic_rows = process_eco_traffic(eco_traffic_raw, eco_sites_raw)

if ECO_DEBUG:
    print("\n=== ECO_DEBUG: raw ADT payload (type / size) ===")
    tr = eco_traffic_raw
    if isinstance(tr, list):
        print(f"  type: list | len: {len(tr)}")
    elif isinstance(tr, dict):
        print(f"  type: dict | keys: {list(tr.keys())[:20]}")
    else:
        print(f"  type: {type(tr).__name__}")
    if isinstance(tr, list) and tr:
        print("\n  [0] JSON (first site/group, truncated):")
        try:
            s0 = json.dumps(tr[0], indent=2, default=str)
            print(s0[:6000] + ("\n  ... [truncated]" if len(s0) > 6000 else ""))
        except Exception as e:
            print("  (json error)", e)
    if traffic_rows:
        print("\n=== ECO_DEBUG: first merged traffic row — numeric paths (top 15 by score) ===")
        r0 = traffic_rows[0]
        pths = collect_numeric_paths(r0, "merged", 0, 14)

        def _sc(path, val):
            pl = path.lower()
            s = 0
            if any(x in pl for x in ("averagedailytraffic", "average_daily_traffic", "averagedaily", "dailyaverage")):
                s += 120
            elif "average" in pl and "daily" in pl:
                s += 100
            elif "adt" in pl:
                s += 90
            elif "average" in pl or "mean" in pl:
                s += 60
            elif "daily" in pl:
                s += 50
            elif "traffic" in pl or "statistic" in pl:
                s += 25
            elif "count" in pl or "total" in pl:
                s += 15
            if pl.endswith(".siteid") or pl.endswith(".id"):
                if val > 50000:
                    s -= 200
            return s

        pths.sort(key=lambda pv: (-_sc(pv[0], pv[1]), -abs(pv[1])))
        for path, val in pths[:15]:
            print(f"  score={_sc(path, val):4d}  {val:>14.4g}  {path}")
        print("  → _eco_adt(first row) =", _eco_adt(r0))

if len(traffic_rows) == 0 and eco_traffic_raw is not None:
    print("Note: no traffic rows parsed — raw shape for debugging:")
    print("  type:", type(eco_traffic_raw).__name__)
    if isinstance(eco_traffic_raw, dict):
        print("  dict keys:", list(eco_traffic_raw.keys())[:25])
    if isinstance(eco_traffic_raw, list) and eco_traffic_raw:
        el0 = eco_traffic_raw[0]
        print("  list[0] type:", type(el0).__name__)
        if isinstance(el0, dict):
            print("  list[0] keys:", list(el0.keys())[:30])

combined = combine_eco(locations, traffic_rows)

active_combined = [
    c
    for c in combined
    if activity_by_site.get(c["id"], {}).get("is_active") is True
]

df_eco = pd.DataFrame(combined)
df_eco_active = pd.DataFrame(active_combined)

print(f"Sites (processed): {len(combined)}")
print(f"Traffic rows (parsed): {len(traffic_rows)}")
print(f"Active sites (counter_activity): {len(active_combined)}")
print("\nSample — all sites (head):")
display(df_eco.head(8))
print("\nActive sites — total ADT (pedestrian+bike lastMonth) summary:")
if not df_eco_active.empty:
    display(df_eco_active[["id", "name", "total_average_daily_traffic"]].sort_values("total_average_daily_traffic", ascending=False).head(12))



=== ECO_DEBUG: raw ADT payload (type / size) ===
  type: list | len: 108

  [0] JSON (first site/group, truncated):
{
  "siteId": 100000425,
  "travelMode": "pedestrian",
  "value": 509.8387
}

=== ECO_DEBUG: first merged traffic row — numeric paths (top 15 by score) ===
  score= 120           509.8  merged.averageDailyTraffic
  score=   0           509.8  merged.value
  score=-200           1e+08  merged.siteId
  → _eco_adt(first row) = 509.8387
Sites (processed): 88
Traffic rows (parsed): 108
Active sites (counter_activity): 59

Sample — all sites (head):


,id,name,latitude,longitude,travelModes,traffic_by_mode,total_average_daily_traffic
0,100000425,Glenageary,53.281410,-6.123190,"[bike, pedestrian]","{'pedestrian': {'siteId': 100000425, 'travelMo...",696.4839
1,100001297,Westmoreland WEST old,53.346034,-6.259275,[pedestrian],"{'pedestrian': {'siteId': 100001297, 'travelMo...",0.0000
2,100001484,O'Connell St/Pennys,53.348790,-6.259690,"[pedestrian, undefined]","{'pedestrian': {'siteId': 100001484, 'travelMo...",0.0000
3,100001485,O'Connell st/Princes st North,53.348950,-6.260040,[pedestrian],"{'pedestrian': {'siteId': 100001485, 'travelMo...",0.0000
4,100001486,Westmoreland EAST old,53.346045,-6.258960,[pedestrian],"{'pedestrian': {'siteId': 100001486, 'travelMo...",0.0000
5,100001487,Dawson Street old,53.342176,-6.258023,[pedestrian],"{'pedestrian': {'siteId': 100001487, 'travelMo...",0.0000
6,100001488,Liffey Street old,53.346725,-6.263270,[pedestrian],"{'pedestrian': {'siteId': 100001488, 'travelMo...",0.0000
7,100001489,Mary st/Jervis st,53.348650,-6.266850,[pedestrian],"{'pedestrian': {'siteId': 100001489, 'travelMo...",9745.4840



Active sites — total ADT (pedestrian+bike lastMonth) summary:


,id,name,total_average_daily_traffic
8,100006267,Grafton Street/CompuB,54134.3240
40,100063163,Grand Canal st upp/Clanwilliam place/Google,27607.8710
4,100001491,Capel st/Mary street,25052.3550
34,100050525,College Green/Church Lane,17656.6130
12,100007778,O'Connell St/Parnell St/AIB,16379.1610
10,100006286,D'olier st/Burgh Quay,15906.2580
42,100063166,Richmond st south/Portabello Harbour inbound,12082.9030
11,100007106,Talbot st/Murrays Pharmacy,11159.4510
3,100001489,Mary st/Jervis st,9745.4840
22,100030847,Westmoreland Street West/Carrolls,8570.5160


### Eco-Counter — time series for one site

Same URLs as `src/routes/api/eco-counter/timeseries/+server.js` (raw daily 30d, weekly year, monthly 3y).

In [5]:
def format_date_eco(t: datetime) -> str:
    return f"{t.year:04d}-{t.month:02d}-{t.day:02d}"


def fetch_eco_timeseries(site_id: int):
    t = datetime.now(timezone.utc)
    dd = format_date_eco(t)
    t30 = t - timedelta(days=30)
    dd2 = format_date_eco(t30)
    t90 = t - timedelta(days=90)
    dd3 = format_date_eco(t90)
    t364 = t - timedelta(days=364)
    dd4 = format_date_eco(t364)
    t3y = t - timedelta(days=364 * 3)
    dd5 = format_date_eco(t3y)
    base = "https://api.eco-counter.com/api/v2"
    url = (
        f"{base}/history/traffic/raw?siteId={site_id}&include=&startDate={dd2}&endDate={dd}"
        "&startTime=00%3A00&endTime=00%3A00&granularity=P1D&gapFilling=false"
        "&travelModes=bike&travelModes=pedestrian"
    )
    url3 = (
        f"{base}/history/traffic/aggregated?siteId={site_id}&include=&startDate={dd4}&endDate={dd}"
        "&startTime=00%3A00&endTime=00%3A00&granularity=P1W&groupBy=travelMode&gapFilling=false"
        "&travelModes=pedestrian&travelModes=bike"
    )
    url4 = (
        f"{base}/history/traffic/aggregated?siteId={site_id}&include=&startDate={dd5}&endDate={dd}"
        "&startTime=00%3A00&endTime=00%3A00&granularity=P1M&groupBy=travelMode&gapFilling=false"
        "&travelModes=pedestrian&travelModes=bike"
    )
    h = eco_headers()
    rh = SESSION.get(url, headers=h, timeout=120)
    rh.raise_for_status()
    hourly_30days = rh.json()
    rw = SESSION.get(url3, headers=h, timeout=120)
    rw.raise_for_status()
    weekly_year = rw.json()
    rm = SESSION.get(url4, headers=h, timeout=120)
    rm.raise_for_status()
    monthly_3years = rm.json()
    return {"hourly_30days": hourly_30days, "weekly_year": weekly_year, "monthly_3years": monthly_3years}


sample_site = active_combined[0]["id"] if active_combined else (combined[0]["id"] if combined else None)


def _parse_ts(ts):
    if not ts:
        return None
    s = str(ts).replace("Z", "+00:00")
    try:
        return datetime.fromisoformat(s)
    except ValueError:
        return None


def summarize_eco_flow_series(name, payload):
    """Overview when API returns a list of {travelMode, direction?, data: [...] } flows."""
    print(f"\n=== {name} ===")
    if payload is None:
        print("  (null)")
        return
    if isinstance(payload, dict):
        print(f"  dict keys: {list(payload.keys())[:20]}")
        return
    if not isinstance(payload, list):
        print(f"  type={type(payload).__name__!r}")
        return
    print(f"  flows: {len(payload)}")
    rows = []
    for i, flow in enumerate(payload):
        if not isinstance(flow, dict):
            rows.append({"flow_idx": i, "note": type(flow).__name__})
            continue
        data = flow.get("data") or []
        first = data[0] if data else {}
        last = data[-1] if data else {}
        rows.append(
            {
                "travelMode": flow.get("travelMode"),
                "direction": flow.get("direction"),
                "points": len(data),
                "first_ts": first.get("timestamp") or first.get("period"),
                "last_ts": last.get("timestamp") or last.get("period"),
            }
        )
    display(pd.DataFrame(rows))


def process_eco_time_series_like_app(time_series_data):
    """Mirrors processEcoCounterTimeSeriesData (eco-counter-processor.js) for hourly_30days."""
    hourly_data = time_series_data.get("hourly_30days")
    if not hourly_data or not isinstance(hourly_data, list):
        return None
    hourly_totals = {
        "pedestrian": [{"total": 0, "count": 0} for _ in range(24)],
        "bike": [{"total": 0, "count": 0} for _ in range(24)],
    }
    for flow in hourly_data:
        travel_mode = flow.get("travelMode")
        data = flow.get("data") or []
        if travel_mode not in hourly_totals:
            continue
        for interval in data:
            ts = interval.get("timestamp")
            counts = interval.get("counts")
            if counts is None and isinstance(interval.get("traffic"), dict):
                counts = interval["traffic"].get("counts")
            if counts is None:
                counts = 0
            dt = _parse_ts(ts)
            if dt is None:
                continue
            hour = dt.hour
            hourly_totals[travel_mode][hour]["total"] += counts
            hourly_totals[travel_mode][hour]["count"] += 1
    daily_averages = {}
    for mode in ("pedestrian", "bike"):
        daily_averages[mode] = []
        for h in range(24):
            hd = hourly_totals[mode][h]
            avg = (hd["total"] * 4) / 30 if hd["count"] > 0 else 0.0
            daily_averages[mode].append({"hour": h, "average_daily_count": round(avg, 4)})
    summary = {
        "total_pedestrian_intervals": sum(h["count"] for h in hourly_totals["pedestrian"]),
        "total_bike_intervals": sum(h["count"] for h in hourly_totals["bike"]),
    }
    return {"hourly_averages": daily_averages, "summary": summary}


if sample_site is not None:
    eco_ts = fetch_eco_timeseries(int(sample_site))
    print(f"Eco time series for site_id={sample_site}")
    for key in ("hourly_30days", "weekly_year", "monthly_3years"):
        summarize_eco_flow_series(key, eco_ts.get(key))

    processed = process_eco_time_series_like_app(eco_ts)
    if processed:
        print("\n--- Hourly averages (app-equivalent, last 30d raw) ---")
        print("Summary:", processed["summary"])
        df_p = pd.DataFrame(processed["hourly_averages"]["pedestrian"])
        df_b = pd.DataFrame(processed["hourly_averages"]["bike"])
        df_p = df_p.rename(columns={"average_daily_count": "ped_avg_daily"})
        df_b = df_b.rename(columns={"average_daily_count": "bike_avg_daily"})
        hourly_tbl = df_p.merge(df_b, on="hour").sort_values("hour")
        display(hourly_tbl)
        peak_p = hourly_tbl.loc[hourly_tbl["ped_avg_daily"].idxmax()]
        peak_b = hourly_tbl.loc[hourly_tbl["bike_avg_daily"].idxmax()]
        print(
            f"Peak pedestrian hour (avg daily count): {int(peak_p['hour']):02d}:00 → {peak_p['ped_avg_daily']:.2f}"
        )
        print(
            f"Peak bike hour (avg daily count): {int(peak_b['hour']):02d}:00 → {peak_b['bike_avg_daily']:.2f}"
        )
    else:
        print("\n(hourly_30days missing or not a list of flows — skipped app-style hourly averages)")
else:
    print("No site id available for time series demo.")


Eco time series for site_id=100000425

=== hourly_30days ===
  flows: 6


,travelMode,direction,points,first_ts,last_ts
0,bike,in,719,2026-03-14T00:00:00Z,2026-04-12T23:00:00+01:00
1,bike,out,719,2026-03-14T00:00:00Z,2026-04-12T23:00:00+01:00
2,pedestrian,in,719,2026-03-14T00:00:00Z,2026-04-12T23:00:00+01:00
3,pedestrian,out,719,2026-03-14T00:00:00Z,2026-04-12T23:00:00+01:00
4,bike,in,0,None,None
5,bike,out,0,None,None



=== weekly_year ===
  flows: 2


,travelMode,direction,points,first_ts,last_ts
0,pedestrian,None,52,2025-04-14T00:00:00+01:00,2026-04-06T00:00:00+01:00
1,bike,None,52,2025-04-14T00:00:00+01:00,2026-04-06T00:00:00+01:00



=== monthly_3years ===
  flows: 2


,travelMode,direction,points,first_ts,last_ts
0,pedestrian,None,34,2023-04-01T00:00:00+01:00,2026-04-01T00:00:00+01:00
1,bike,None,37,2023-04-01T00:00:00+01:00,2026-04-01T00:00:00+01:00



--- Hourly averages (app-equivalent, last 30d raw) ---
Summary: {'total_pedestrian_intervals': 1438, 'total_bike_intervals': 1438}


,hour,ped_avg_daily,bike_avg_daily
0,0,6.5333,3.6000
1,1,0.9333,0.8000
2,2,2.1333,1.4667
3,3,0.9333,1.0667
4,4,0.8000,0.6667
5,5,4.6667,0.4000
6,6,15.3333,5.0667
7,7,48.2667,20.6667
8,8,137.3333,58.1333
9,9,137.4667,51.8667


Peak pedestrian hour (avg daily count): 11:00 → 267.33
Peak bike hour (avg daily count): 12:00 → 65.07


## Vivacity — static markers + metadata merge

Same as `+page.server.js`: attach `countlines` from `hardware/metadata` per `sensor_id`.

In [6]:
if not VIVACITY_KEY:
    vivacity_markers_enriched = []
    vivacity_metadata = None
else:
    vivacity_markers = load_json(STATIC / "vivacity_markers.json") or []
    r = SESSION.get(
        "https://api.vivacitylabs.com/hardware/metadata",
        headers={"Accept": "application/json", "x-vivacity-api-key": VIVACITY_KEY},
        timeout=120,
    )
    r.raise_for_status()
    vivacity_metadata = r.json()

    def merge_markers(markers, meta):
        out = []
        for m in markers:
            sid = str(m.get("sensor_id"))
            sensor = (meta or {}).get(sid) if meta else None
            countlines = []
            if sensor and sensor.get("view_points"):
                for vp in sensor["view_points"].values():
                    for cid, cdata in (vp.get("countlines") or {}).items():
                        countlines.append(
                            {
                                "id": cid,
                                "name": cdata.get("name"),
                                "description": cdata.get("description"),
                                "direction": cdata.get("direction"),
                            }
                        )
            row = {**m, "countlines": countlines}
            out.append(row)
        return out

    vivacity_markers_enriched = merge_markers(vivacity_markers, vivacity_metadata)
    df_viv_markers = pd.DataFrame(
        [{**{k: v for k, v in x.items() if k != "countlines"}, "n_countlines": len(x.get("countlines") or [])} for x in vivacity_markers_enriched]
    )
    print(f"Vivacity markers: {len(vivacity_markers_enriched)}")
    display(df_viv_markers.head(10))

Vivacity markers: 18


,name,sensor_id,lat,long,pedestrian_total,cyclist_total,n_countlines
0,Georges St Upper path LHS,2158,53.290352,-6.131580,147896,9448,3
1,george St path LHS,2159,53.293072,-6.137990,256153,7305,3
2,St Georges Upper St path LHS,3763,53.292042,-6.135690,315259,9253,3
3,Main St path LHS,3771,53.289040,-6.243410,228203,15610,4
4,George's St Lower path,4158,53.294338,-6.140740,113091,8937,3
5,Main St path LHS,4159,53.301659,-6.177760,261025,12992,4
6,DCU road outbound,7487,53.385639,-6.255570,152387,6995,5
7,East Pier path LHS,8479,53.293316,-6.129289,130478,2515,2
8,Drynam Heath road,9510,53.442619,-6.195500,6549,607,6
9,Ongar Distributor Rd road,9646,53.392021,-6.438760,19298,2400,2


### Vivacity — countline counts (dashboard snapshot)

Matches `+page.server.js`: last 7 days, 1h bucket, example countline `22988` (same hard-coded probe as the app).

In [7]:
def vivacity_iso(d: datetime) -> str:
    return d.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")


if VIVACITY_KEY:
    to = datetime.now(timezone.utc)
    to = to.replace(minute=0, second=0, microsecond=0)
    from7 = to - timedelta(days=7)
    from7 = from7.replace(minute=0, second=0, microsecond=0)
    url = (
        "https://api.vivacitylabs.com/countline/counts"
        f"?countline_ids=22988&from={vivacity_iso(from7)}&to={vivacity_iso(to)}"
        "&time_bucket=1h&fill_zeros=true"
    )
    r = SESSION.get(
        url,
        headers={"Accept": "application/json", "x-vivacity-api-key": VIVACITY_KEY},
        timeout=120,
    )
    r.raise_for_status()
    vivacity_snapshot = r.json()
    print("Snapshot keys (by countline id):", list(vivacity_snapshot.keys())[:8])
    k0 = next(iter(vivacity_snapshot.keys()), None)
    if k0 and isinstance(vivacity_snapshot[k0], list):
        print(f"Hours for countline {k0}: {len(vivacity_snapshot[k0])}")
else:
    vivacity_snapshot = None
    print("Skipped — no VIVACITY_API")

Snapshot keys (by countline id): ['22988']
Hours for countline 22988: 168


### Vivacity — time series + aggregation

Mirrors `api/vivacity-counter/timeseries/+server.js`: parallel hourly (7d) and daily (24h bucket, ~last 30 days window in app code), then `aggregateVivacityData` over clockwise / anti_clockwise.

In [8]:
def aggregate_vivacity_data(data):
    if not data or not isinstance(data, dict):
        return data
    keys = list(data.keys())
    if not keys:
        return data
    first = data[keys[0]]
    if not isinstance(first, list):
        return data

    out = []
    for time_entry in first:
        agg = {
            "from": time_entry.get("from"),
            "to": time_entry.get("to"),
            "pedestrian": 0,
            "cyclist": 0,
            "car": 0,
            "bus": 0,
            "agricultural_vehicle": 0,
            "cargo_bicycle": 0,
            "dog": 0,
            "electric_hackney_cab": 0,
            "emergency_car": 0,
        }
        for ck in keys:
            series = data[ck]
            match = next((e for e in series if e.get("from") == time_entry.get("from")), None)
            if not match:
                continue
            for direction in ("clockwise", "anti_clockwise"):
                block = match.get(direction) or {}
                for vt, c in block.items():
                    c = c or 0
                    if vt in agg:
                        agg[vt] += c
                    else:
                        agg[vt] = agg.get(vt, 0) + c
        out.append(agg)
    return out


def fetch_vivacity_timeseries(countline_ids):
    ids = countline_ids if isinstance(countline_ids, list) else [countline_ids]
    param = ",".join(str(x) for x in ids)
    to = datetime.now(timezone.utc)
    to = to.replace(minute=0, second=0, microsecond=0)
    from7 = (to - timedelta(days=7)).replace(minute=0, second=0, microsecond=0)
    from3m = datetime.now(timezone.utc) - timedelta(days=30)
    from3m = from3m.replace(hour=0, minute=0, second=0, microsecond=0)
    to_today = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
    from7_iso = vivacity_iso(from7)
    from3_iso = vivacity_iso(from3m)
    to_iso = vivacity_iso(to)
    to_day_iso = vivacity_iso(to_today)
    base = "https://api.vivacitylabs.com/countline/counts"
    url_h = f"{base}?countline_ids={param}&from={from7_iso}&to={to_iso}&time_bucket=1h&fill_zeros=true"
    url_d = f"{base}?countline_ids={param}&from={from3_iso}&to={to_day_iso}&time_bucket=24h"
    h = {"Accept": "application/json", "x-vivacity-api-key": VIVACITY_KEY}
    rh = SESSION.get(url_h, headers=h, timeout=120)
    rd = SESSION.get(url_d, headers=h, timeout=120)
    rh.raise_for_status()
    rd.raise_for_status()
    hourly = rh.json()
    daily = rd.json()
    return {
        "hourly_7days": aggregate_vivacity_data(hourly),
        "daily_3months": aggregate_vivacity_data(daily),
        "countlineIds": ids,
        "dateRange": {
            "hourly": {"from": from7_iso, "to": to_iso},
            "daily": {"from": from3_iso, "to": to_iso},
        },
    }


if VIVACITY_KEY and vivacity_markers_enriched:
    demo = vivacity_markers_enriched[0]
    cl_ids = [c["id"] for c in demo.get("countlines") or []]
    if not cl_ids:
        cl_ids = ["22988"]
    vts = fetch_vivacity_timeseries(cl_ids)
    df_h = pd.DataFrame(vts["hourly_7days"])
    df_d = pd.DataFrame(vts["daily_3months"])
    print("Aggregated hourly rows:", len(df_h), "| daily rows:", len(df_d))
    print("Countlines used:", vts["countlineIds"])
    if not df_h.empty:
        print("\nHourly — pedestrian / cyclist totals (sum over window):")
        print(df_h[["pedestrian", "cyclist"]].sum())
    if not df_d.empty:
        print("\nDaily (last window) — pedestrian / cyclist totals:")
        print(df_d[["pedestrian", "cyclist"]].sum())
    display(df_h.head(5))
    display(df_d.tail(5))
elif not VIVACITY_KEY:
    print("Skipped — no VIVACITY_API")
else:
    print("No enriched markers to derive countline IDs.")

Aggregated hourly rows: 168 | daily rows: 30
Countlines used: ['22997', '22998', '22999']

Hourly — pedestrian / cyclist totals (sum over window):
pedestrian    35221
cyclist        1838
dtype: int64

Daily (last window) — pedestrian / cyclist totals:
pedestrian    151824
cyclist         8193
dtype: int64


,from,to,pedestrian,cyclist,car,bus,agricultural_vehicle,cargo_bicycle,dog,electric_hackney_cab,...,pushchair,rental_bicycle,rigid,small_van,taxi,towed_trailer,tractor,truck,van,wheelchair
0,2026-04-06T10:00:00.000Z,2026-04-06T11:00:00.000Z,305,26,312,18,0,0,0,0,...,0,0,1,0,1,0,0,0,22,0
1,2026-04-06T11:00:00.000Z,2026-04-06T12:00:00.000Z,354,14,402,16,0,0,0,0,...,0,0,2,0,3,0,0,0,34,0
2,2026-04-06T12:00:00.000Z,2026-04-06T13:00:00.000Z,400,11,408,16,0,0,0,0,...,0,0,0,0,3,0,0,0,21,0
3,2026-04-06T13:00:00.000Z,2026-04-06T14:00:00.000Z,423,22,423,16,0,0,0,0,...,0,0,1,0,3,0,0,0,21,0
4,2026-04-06T14:00:00.000Z,2026-04-06T15:00:00.000Z,479,17,471,14,0,0,0,0,...,0,0,2,0,3,0,0,1,21,0


,from,to,pedestrian,cyclist,car,bus,agricultural_vehicle,cargo_bicycle,dog,electric_hackney_cab,emergency_car,emergency_van,motorbike,rigid,taxi,truck,van,minibus
25,2026-04-08T00:00:00.000Z,2026-04-09T00:00:00.000Z,5513,309,5693,295,0,0,0,0,9,5,181,54,23,16.0,617,NaN
26,2026-04-09T00:00:00.000Z,2026-04-10T00:00:00.000Z,5033,265,5947,342,0,0,0,0,5,6,179,65,43,7.0,624,NaN
27,2026-04-10T00:00:00.000Z,2026-04-11T00:00:00.000Z,4551,262,6053,349,0,0,0,0,0,4,191,63,55,2.0,603,1.0
28,2026-04-11T00:00:00.000Z,2026-04-12T00:00:00.000Z,5393,261,5970,337,0,0,0,0,7,2,189,16,70,1.0,330,NaN
29,2026-04-12T00:00:00.000Z,2026-04-13T00:00:00.000Z,4719,187,5370,254,0,0,0,0,4,5,165,11,49,NaN,235,NaN
